# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrNfWlz28a24Hf+in5IpQIkFExKtuPwhamxI9nxjLw820nuXF0VCiSaJCxsAUBKtMvzt97398vmLN1AY6Nkx29qVLaEpdfTZz+nG5ZlvUjLNJ+JFfyxC0cc/SJiP4p8cXIq7N+2yX7rJyenR8finjiOd/S+lDflNvfx8kkkk0Dm7mj0e5GKQIpcLrd5kRazkRDfi3dpkBYigv9xGkj8+9dWBn4iln6+9vGdTAT09ez178LHi538IOxM5qswEi8e/+P5i1fiHrQkxOPzd6+csfCLUCSp2Ph7IQso52MLZS7FNvFpCmKPzRThehvCC+nSMM6ho0zCqEQhRQZDxN4T1febxy+EvfSXG4njz/x1mECrcHkeJtsbR0RyD3NMaRQ4Az9fbsIdDz3zcz+Cec2EajSHgai5itLPAxiKXG+hesEjOYtEmfsL/z28lnGWQnWx87Gpe4Hc3Ss2sbCDsFim+AhG5qgJMIho9Gkp11Is0wS7WvpJAHD8d70Qwv4Vyv1AVaFMDpCRCQ1dD5VqmiASa5kAIJdhmtSrUcMtl8U2KmmtoPv1FidFBRHiDDYeqoDZwMIUG4RdpFejBtv7bVHKAiGVyTLMsUgcFrEqKAsRJkXpJ/BPpu7IsqxRiAAqccCAUzmM1V1tAfGgKCzQcqXfr5fVVZQu9DWOJAqr2zDVV++LNNHXaaGvcqmvis22DKPqbl8VKbYLgP5SFtWTcpNLPwiTdfUgjKt2PoQZYLEc6ftkG2d7HHmSjVZ5GovXz8+Fevc89tdyzH9eZcWIC7iihsAqXI9Gv529ORNzGLWb+eXGDcI88WNp63t/UeBf2/OwY89znBEspXf+6tf/BbWqwbrn6fLKdkZ/PD/78y28sC3oLSmtsbAW/vIK/0ZyRfd5uN6UljMajb4RbwGR8xCwbQbD8hFpCrEOgY5gVUUqVn5U4ORSkQGJA0qU23gB9CAjwJt8FwZpPqL5uUDX3vMXj5+dea+f/+PsHIfw8L43mUzw/wjfvn1+ivM8ntx/BF0HciUKfyU96tYO47UzI7QCHPnDj2BIQPRJmgPbCj/4hJc8QFwxfGuXfuwn6RgJE36vsGiZOi7iGLYTrqA6oEjBGLikPtRquPRbdYg/uR8CHUC/W3mW52luW0DTiifg7AEYxgh2ND6AIFa9HosNTAsad4vwg9R9X4ufxcmxQOqhq0N9nVctS+RRMfBDIE1gbsBXoVc7DpMwTrG57MZR3VIX30Pjv4j+BfiCDtc5MB5ZdxD7NzZOz4FO9ArWzcKM1byXaba3HfOFW24AURI/jGxb1xxXbTh6Gc4fv/z1n6/eVh1iTeSzvHIJIPGbZ08QZeHPY8vp7zvZyby0VREqkUtgKAm+Voi2jKSfbDMv3ZbZtizsKymz+aMJ4E6YeKtcSm+9mE8nNQY+ASbrE4czOCWASexC+R6ZZkjSqgKdopXSF8TnKyQs83096KCAMRdA+jKwA0RYEeAkkb+5+Ksi+fdpmNjMHtxXv7/zTp+/ARh8bzkOAkmXCgtgFXbg1IDHmWAXxOuAkRRX3rZA6mq25bhUEET/99+fTKra1xvgLzhIQAJhg+SxgwJXHoGFeEyVfjZBZiwIzQ/6DmBwaWZPnMYbNaI8BqEu7WAswjWQtvQkomMxf5dvZbOCMU15ExawZAHIP8sF9mu1esUfKJvLON1Js1ij1JeDRt4sZVaKM/oD4rTuPfNBaDCC/QVSt/SidF3YNRK9WiLmCH8XFqwjIScBlrcEdiRARahQC3hZCrzGBfJDgQHtpkiW1E4QrrYFdAsk8/LpK5TCm/1JAKTrkPCnZiW2tQYw5amIt7JA1alIQSuAF/4OeV/NFVn2wEjXKOGMR9d+nsCjgpmaunFB5pQy17e2xQsHJBmDxAT4zS33e2Pw7veaK97awNIHlSfN9/OnJP7/5BJcG2kDhSDzADVh5APq0i02ftbzSLaegU6LTwyMURN317I8h0uZ29iP4xZwL3cysnWBPx+/efn85TOnS8YkxWFZYF1kXriIUEULrqgQBHBdkz7cYB8e8KpFWoTlnnFfcc1bcKx/CKByJgWKvUOjKM1RlH9rFKPRMoK/4iyBxiW/RNz3PBBPpefZhYxWY9RlV/OXaQKIDN3NszxMSmMBsJCbASliOWQrigADWQJhGFKECkILUNScAz0GYl/gC+ym+YawYPiNF+8OvCwPVhx4nfkwweE33lWYIGNU9Fe9fJ8uUBx8tPxliYaHNRMgjyyZeMs08vHuU7OCByYOSJ9rqPQUtLK6r29AQUWrAwSUDEKURTZegCKe7sJIsQlgCiDxAdpg+JVKVjXbl8CqcDykSEH/j1wcj7IK4f7kAT1YsEkCDx5O3M4YoREPubahz/YLMzVkF/V2g1U3sNwcmrvNAmAYNpYHrPADO81AQjV7dQxZOIzMDbKq+mCjBwDwqY1qfrDvwJxeKVB4WZpGUGK5ct+RNv4a7s9uwGQGxmiDDuVdp/kVEOn8BBQ2xowLSz2zLushAzhi/wqWLi86ACMp6KVXSlbWxCeTArgnAcKgMmD3b0Nk/8s0CNcpGnmAZ1vUpZWtaFjve7+y3IEtp3EWIS7Am6o5v8zDBUwHLIGtRCuZcCpBcxesTJJBuQQusAx9MgELiSKrEjpaOeDV8oJwCeyifoMsCebigzC0LUWJwMWRoJyDpZAmbylY0yDKBWuoGFIjFDhEjUNVNV1CdcKRoXIAU+rh8+hrsFeN89Dm51DaUHuE/ji+wR4BBbL0FmiXx2GnBKhyAYpb2zLpxQJVtqCSLR3yolns8u8Q1ojZ49Hf+GHKqMgN1hqZj3ct0YYulMjLJQB4DtOXoGol5T3DwWbRGncexztYiga5noPaWVaOtcodhc4r8nERxQXyvZ9HlYPtzeMXDRojtWCzJeG/8pfS22wXWicolj4gM64y0F9eVwGdH+XQxeUwD0ZtLEdVrNmG7bg08a46Dkue0zsvJAtnoFjVttxRKRcJCRnHQNl6vD/AgEHLtVcu+USI91NTK+5uR8+Ly0FpgKqR7JE0oGnY1kvil9kWFPNISmJolSfxt6czWNKWtaLszcmoJgvAlTywaWDNbhIA9qTx5DoEeUnSDIujf2YBXSy2q5UE3Wk9nzg42lUXJmywoUDoh9cCumIBZj+8L37+WRy3zDJjvdDgXgyDXc0w6S2Q4HqgvbhwagigSoSOMxd/GWodzbafoqcPaabypjmOMi19lK/FNrblDYjIzM7BhKQVdrr6or2yXjPFKO+tckgDsczER25M2Xczd7r6JJ49wddWd2or66MxA3Ekypk7WX0qDCmi4EKtGkKZxVMWZtLgEGP0XI0FVFikTkuXbxsxmmorlnEavnsapdcv/BIYQ7J+DU1HoaFwYl8Ao9vKu9gbSiwwHuA2sFt4rEa5SiPgwHMaL1g5HjrqgIsVXWMdUIcm1Fyyb8RTMBU2f5wCyAOJOsgqXFIogKMQ/2c6uSF3Su5nofLCD7IfPT1XJj7IBm+FTe+C2I6Xnh+t07kVL1vWviL3d/uMPV7dBnt7GeypSzZ3YChdxiIqsKDuFIRFliYhdNTDUu7Y/me3rRAWZ1njK+nTjKjXYB6AXkWrrO5IhTIf0IKbD/5CxbLck9XXlGq/IgGayqYr3mFtYT9Qbo4fqnE76IwElRX9uRR2gKGBSWigFsUfQHsShR9RgKIKw4hfAUQYTGEXpij9eBFCAQpMae0WGq+aUtEcdjDDaCY8HG0tqVH8So7eoCFkdVAAnR41tyMXCDraQXcp3OU28IFRlfE2cgHh02uvXJ0cA30i0A5USpLPLb4A3WIDht1Vu3SDrbC9rEb+BFpY5ykA7g35zHKTmM317NfQGiXmldoF4/bSxFtnW9DZ0H2Tlga23C4XoG/DtO/tuWH6d2bRbKvumtyZDSN/uHGiJMJZwCphxkkV1jL6EdfyHdd1W3ynl6u0HAxsBBkSol9tHBTFQz/WRlUPjoKwPNodH02OCADAALoO1i/gLwyC2xhXD4o08LKNY83V+aK14VVBjyJRa8+6NFxDd1yAPoB2bTw9nXhHMxlGO/IffcHsoGGFezs/DykCUZR+8QXIpzxYd5o+2Cefj4G9SBjvvi4W1gA5hImtZbEbKAfcaWBhnO6qfeGaNVfrEG7evmz/vYvWt2RdPP8bq9VYpzGaVtsChSxFKKaUKDCweuxarZaEb3vXoyHt2OvfUaFfY/WO4kwj3pIv1r4wYEGdmVz0siVWJMAHzGYHNJj+etZlx4ou0B6nCHZxAG6rPqSi0Qv743fU+3c4mOqSWuQBfccazXefejDtoN47+KLj6O4H6GHLooN0/Sy3YXr026q3q/YGFlm8dApCs4NDoij5V4VA7ySd0eHmdbwAxttv1IPZfPW1zJG7CPPDgQ5TdavoE3W/Lo3WUQIMQbHjE0zL8AMbyahEwkPTId7p4MJKVyu0VqzL2S0D1TYcKfreMtt6qmrLmEOquUXvUYlxzx834SXsJXCwHANZe4zpBqmRI+VYDXaGEOnTlLuOtoxing3+P26LK/OB0+t+y8xlIA5aPTHV0QET2C1T2wKQWU5/LKSh0NGbaz+Pt5nd6415ocIMEYhCdsXc6lbRtb1YovFdGaqqm26w4/fERwPODzD9LAlBRwerjnP3ZhTSCCM2Q69knsCS40r9+vvpY7EHBAjX0ogPFwKdCnlYxux+XW5PX77kHEVoI8th1LnOg/Mxzc9Hvy0mOUbp8m7WonLWNAyDhlJiog4VVkJPG1ZsCpKVGRaev/NheoCRdgsZ2Ni/1fTqOnz7EtlOc/+6SfYxtMd5PIm8Vgk4Y7Clp8djAb8cuJ6MhfrnOJ1ckapZF3/ZYdwq4sooCrNC2vb0x4djcR+aOTmBi+PJxCH/XzS34Xospj/irwfw6/jBg3ZHgLWYcLGOsJ2foPr0J2xoii3+WDf0EB5iRtD0YW8zaHIy1J9RYiX6KwO5C5dybuE6WA7Y+wkomMDcgPW3HK3k9OTqYbKSOcoF4kw29M/P/W2ZLv0CQzHY3FgEJYi4Ob9ExlVOHzr9jjGbcrnmYTzGdESv7qEoZVbMYTXSJSbeeLkEnWuL8mE+PX40oCtiE0vQZ66KueZYfGtdjlVWKQxqvoZWKZvKo3FieCiWxaat3NaIWuyT5SYH1vlB2s5oQPFBFw6oyiA0kZTvwio+y7MvdOMxJsmmIo3DMgy0+lmxGZRNXpmq1MQqrSHOiPPKYD5pch+UECkmTqukOsqY5kUI/Srf+jR8Nwbmvoy20CUFYQ29gFKC+eetZM5lshvMoMYoKyvPrjhLyhCjZCKW71OMzawjytdEZ2/aZEArWLTSL8tcTcQM0t1NmYYKOK+WQg3T6VemB00jqCHskmCl8w+HbHXostazjJ5sgDXpVEfq3b26zNFu6k6PTnViDlwVJZC2DIwOeBWR9wI8+MZxCwxt2s7F7GQyuWylGOrRqMKYXQYcYxtFYpEGe7DrMNk6lwEgSARqh1hUbilL4QroIjZeOB0nLOUnmj5Yj2KsxVoHDdK0nFv3AFNK1CLf4cu3z6wW7vFTYf/x+O07cU/QvXj83JkpXykgCSUmrkIQwyif4EkBD9nyul1Y9SCQGQke1vfULJs+zLaowVAhTFMnfBZ7jmX3aGR7nfaYFJjvOWH4OHcp2AiQYy2YRLGEWZSF5Thd3FfL4GYK64rqSq+QBpWCfi8hUFOLPASdxXC+wv2bF0+eNcVDO11GjbG2JXTAuZdWcAxertyiqv2OMdJo/xpTaqHU0dS9DxBgfY+lmNz5kT3UC+l+jRnf3k+FttRNZ7m0uGsKugGOopGdVElhx7jJBRG8wvQK+y3nVly8q+TQM+/YPd1x9bo6ajY/HDrm7KKKGSilsMENKHudJDozFhDGIEgwubORF80ULGPepnCYstVDltzNyAGxZ0/tjNAlYZkzP1cp+82Iru7RfenHMnjHu2Dy/VN4ZBfb1Sq8mVtulqxRsZGRLOWcHUwU2m6RO6XS+ztprzYuZWe2UrcovUy9G+YsoPJTlqExZBVgX6w9stzmSeb6ee7v7Ysp5t7oX5eglCE5eYks5y0SM5KtwwQsuVandSIyhf5HX6g+9siMAypkawiV1MJBK90QgNFS3vo0RewWc/4Vlh2DEnzQoagR0CtQrZoT8dr6IQkOff2LeMD+qR9dsARgdTHJrLgwxK2OpDMyuu/4r51CGRfUYtAxbVgu6uIETQtcuwJFY7jepuhQxHVML6aXjpkd1+tpAPqBSwzKY55RO2rZ8WYMSTZj6LRJay4QfupGm3OgPfMT06/Rb3A13ZcuZhjj6pQh+TG4FZm7CBZ1p+nyBTx6Q09GA84QlRVmggLRRO9XGa7BHekK+cFOKjjU47FVaphpenCxsTAHM8dfPVwfFw89s+aCOX87q2eJ0WHexcSbiFB0qEy8ti2gvRAtn0NXkVEssqB9Zj1vmrwXf3ZoQHMFdxfmmJ6JXaX53h4yPjmwTGU8jBRjEn0AlpFKaKF0jCHfABPgZMj0ws1+W/KgfNzFLqjQQSNN5h4+bWXPTDh75r/+sy97hjNo0KD7uFYtKIr4bpf7sbdefHeJ1a3Pytn9Gil9eqObmdWXmVKWWbi3WHOyS4M3oFlwgCk0tQQs39p/BsaG0xF3lSOF0sBwL1zbDGnulBv11H2VFbhXZuXRvgBQHFpFNz5gQ5RtfLVxivZZzedqbxX5CCt5iBUvwDQbi5NLNwZFzhE/gziYmBOrgMRV07LuoW9+dfDebm/bMnXwwxu7ePFzuQSCAvoFXEwX7yXtD0ZDLEea5h20Wz/IfWVmx2CHyppdIwB6Z1qV2IPwuym4WJImH2Se2j7IsOmk4dTFlLeborWaN6BD3EzHYg9/91No46ZgCI7pCqQrXO2rZ3v1rAdiBIQ8zWwb29xTu2B/Tqld+GsqBz17EIm8lVCqN/J9DwrO9JHhaf2r34mHSuagA6/4Czg+KAmM0bZdiCNxDUwIGANWhbsN33VtXqgKogAlsT2dHN8HfQt+d/YDkhAmgB/fZ7ZV/GVox1kUll6xkbLU8hyvwc7LUZ7zxtcx7nYdozk+VvtdG2bzabgDe5g2dW7S92Q7cKiWN4AnEv0pBeqOlD+I6UirNAk6vhXq+SBDMDYU5OuF3ySFIcpt7Z6jPjr7WdWskaMwkAweQu9A0xqiswYR4LgqKiCwV+WWtLXBd/1kbzo1C7ku0JlVABBxBH5eUhKxihuPDTGDQY1wLCjFV4LWiSoobruLnI4nYFexEm65J7TR6pGT0MJWREeBips73FRnZ4c5Q9fPAJYBIDh2B/04Dfrva7mvGvIJnK9R+08CKpl5oLI2wIpwLDhaTB4RfISIBuWAsFBnhhX6Ewh54k7uX7b5ERYnZj0jVgnIIjZb4JfFDDPbcFM36MugrBedpGSsTQTkEn3Z1rjtG9GjI/MkhBH8CapA4oyJxcAD5EnVU87HDikfG3cG2olTDxYtN2oq1c43Kp1i6fYY6lrptmxu0NHbBIHpMNt1sIEP0B51MKYRt82jbXmBby+r3DUU/kgBBrNVPJyeIrMis6KTE6EIHFr8KmoJOTFq0wUN11D73FDpH+N+bA/z+ou7pg/bp1IZffIp1FOZcjgRvAm2S7i5Y8bIU7S8ZK6aqIGAI0Miary2HRufd0r1DqdTeFC33mM50GYbL2NEpOqVi7bHW1m2JGoMRBkEHpmUzbLUt4s8EozxgiHtMpA7TeC7MFl7SvUJtoCkqIF7urp9oAqupgfahAcGR4hZDp6ENTtQZRmBJPA2KRArCnC6QFPpeDKZHOjHS3MMbjCieMt0Q0Zk1LYp0PaIMYuUzuBgyDi9dkvLPWQsZ9tYjwmO8gamCK9I04kJlsYD5VKat2jpM8M41RoOuNzUyHocbr15LgemqqcLEAYtcmiluWWGBi+1uwpvPE7GKVr4PVghivRyd8r3wuf4cOICub74CBgOdLFxe9wQYhVPIWGJQsCgAPFL/X7WhwEGI1FkXHMpkPXzmmO1uSYBurVjQjE73E1XVL5Ojl22fZ5qHwWe/gEGb9NEY7cwmjbKg8rWzZdFMXrxRc3eDOArXy2N/UKdvXI55LDtNPiN9qtXqe0pKIOCSZmW8J5EJ5s/o62c6Hn3Ma+hiq2NBnbs8JJ0VZzPzuXT41uh2+FW17bW9msAoZyyP2NlnbY3nBr4bATpWPB9zph4G5UYb/x4NQOlEbWKK6WwUk9uWMoYKJ5VVANTak1EJ5uyGkXtIfFMaz0aMJBzDA/k8sCDy2pHjUowPJwJdOmIf7W1aD0WlaDYGgJtnTPw1mzyQOeNnsvLdipLIm9KO6sTmLivdiJSjR+4qYjAxdktt2Y8GSyroCzmvnrloWorqtmSKIhAKAfBFieXO96DjHrksHqvn4if52ATMUwefFlaV1puZN4acBOmrcF359Kb6KXabSV7NZ4iqPoTvahYT7JXtQ+qjgsOZXXIOCv3vCfUbntxmKxgkGq1CX5NBvn/JkDyN3JsFKdvpNkciJ4oDBrUqbuJOFiLnzrjg7k3oy+KwjjNsM/wwHpzeXoCNCxQeEW/itmj/O6N5Bt9ooE2fBTDR9XyUOjGslSwBc+jQwiT7hNihkaCLp5IbNIcJGeQuuxamlenHvKWno53506BoF4h+9lBIejObk/QEf82b+stGBLRakuPgtwNeXUaNcmUrPAmTV5iRjs/urpkacjn8AycI0dikeQk7vK/cg7sJP9G/MHbIur0qELYq5xOLPwBD3/0I1gF13UEp1uCuo0bwkWUCj8AHVa2bfnKivfJz0kLUdFpob0ieE3ymHkQ3gJqf7H1oZEGncuNfR50wFPAuvaYNzjw5HoMkVsH3xhh7WvgfTRfh/RStfcW6e5K7pvqVZqVpqMB2aA6fhGl0tRg9xWO0OGDTbihkDexo+vzO5gwvtEHv1y5gC7EzZ3hQjuQWIt9iebZxWwmfvrRgOCmcYRMsI2zwsYpjulINg+mr/Y293SkoL9xN/ImCMFqL7GD6fFlraTm20SBj93PlAmmz0ECglGXRDPqmny0fN3193oHndntGNW4q9eoTli2qBsyxdQ1ILEeB6apTY9P7jdb0SJkTgeTQL9za7PFaGkY4EkhG8pImk/dR5iJBSxKzidQsLn4UDSdW7/6oGH5uTXmByj+dNecCKCHBEAuMVjZMyfF0tS241xGodxhj49MRZ/cC+sc/QuRHy8CX6AImQkFisRDH2joRz1Ap5EptwQVBXpea7gFwIIjPSy2L+d6z4/FcokFiHJrzNrHIvEhPTVCmefF9SqUlGrfy4EaOxheVyxk1FZqK3UExc7rC4uuG8cNEWaokuqGi/KNWZY9BFyUr7kkXTdSGyhej8Xwigu1dnVUOKAK1vdcvLo3KylE0fNSd2pmfNcYxqSV625mgBKC2ICsxyDE/mMblpSkSuEdyxwmMC0+EYvk4sxwFysSr2jQ0aesmoWIXM0yXeWLRalZiQi92TBTvFmInpilPvWGpPq5qmnesolhhtNUSKnmQI62fnvND5Oz96tIDXCiQgHiYNRuodI9BtSszgGmTymlVZ9HjBmAJGaN5UPWjudRhktFBvOGu2CuvQbMEuk380T4387c7VebK/5YuxyQScL/ijnyn4o/8p/xAHyQVdJvk01WVzWvVH/7m6lZ6CJNI7u6dap8G/WCb5yalbLdoO6cwdaB0+qWF6kz1nyTnvG1U/NLeqpuHM046fdA+zUj1aNU94YwvtI8DzUWU1cx0eqqQm8+uoes4sbBqfVLwMoLCx4BAxnS+d5UR3XKhA8DQo3uqjeXs9FwzXjaZ9A6rbjWwVO8iJUVZb4idmZ9+7+/jb8NvG9/+/bFt289C/egXvWfI0cKY+vwuNaprHR6b6N3PJEaTz8rym0Qpu6mjCPLof0eRmUUjzhsJSmRF9tK6HJMLpi1nD54Mld7oq/5wC6PwD8emgUWGZwGRQH7/HWzz9Q1d5x12sxgpp5XFun33sfkEyWxOoZAOdPoKI0TFb8Rb9D5TrwpAltzIr7lw0rQksGNHHiMPCaioYriF1Kfko/nbudhmhuHfiBunF2oY+PQNrPPLqpj49hjx7Ss8rsQGc6qU9Soxv0aVp4+BQGjx/Bq4v40wRPs6w7uYZ9GBZboumJV49ZRcDuNE7HKcHllg2G0QAa7xGBbQRlRadZaKzDDH+/85AOx+AUmKYhiC2sTk6EYYKoPcX36PgCDL183rXcC3nRgx1t9ehYpXDAAzJUrMGzYRYsb5SGEiT8Yg43e3JA0xZlS0izNaNpjnlQqh4+rt4BaPsbJb2j+PRsYcDjXflhCjw+cJgQzsMBkF4SrpM25oI3G6fBnOwBaCwKUR10X4RO57BKzpco5rtVYwHUxH1gyoHAfNJCk5/hmoB7KfbDvEHtTXHOVtAr35ldX8Ck6IV7VL1GtAbXqsNWL6jTHS3SCT01poQ/UB2a/vMKEylbmoaEzngBjPEswL8af6Y9cAM2igYMxm1SdLGS1HJO6hx4B0x7ckTm4Vil9PGVrCoPATXbq4LQG6yOWafDLHWpWRZ9lTTrUEAVp/yi5AwPlzycEBTgBqSieMTa4y12zDSz2AqNeXh36dHJqIf1Y7ZNQLDKcdrV7xxrYZN/9oeb+6z/5QBGrFf2AZu46XJZ7M/OYjkOxri85WaWldLbinz0rV3H6FsMqhvlTtWDWeRhnIYEfMytzXgl/vX0v87Sw+uKXwAf5bLfqAFmZ8MYzwxVro8cJ066VTqg2zD9/7BwMsepcGEa2plJRp3OtUo/6m7ej2Hc/BONgMo0R6+6tvMEQIg1beR3NQPrPc/Fwgj+MW7fFzbmsM9CPK29wRE1NhTQ9i9aA4OCuo4XlfLWT63hxCcBmMkEfKhw7GB7OaXq42TDbYnYBLHSBG1t9/GpPAPULYdNha9qbih//AdUIcCd1bsVRUkQmxGUsAiRuhyWd4GOFCbPxJ+j2I2cvwDV1bvUeL9HNe+jJefgqWHLHxIkBzKUx9q0+66mWny9DWKehtR/U+PuqtxT+huJvM6JZXMZcfCDszBfTWX3Y4x7Pgwbt9vOFS61VzobMRXwZwCiulJ7K8qeBJNMxa7EkhyqV9TOZsPU6TEotj0z///PHn3voUkNY6GhXO871d2SE/qEQF8NQWdHMfIiB9jCG3pNQBuA8lFfCoofL3Cp4dLuHMJrVKRXRCj9g7O4LkXuopcN4rhGmiekaz49nhAS0nfDuYmZwR9CdA+x32zHT2DlzN926T9VsKKSGpYhG5IOm8fiDeqS0j1GXktkTQ2jU7L5hTxpN1w/rxuHZyLC19WfSVluZc5xXfUENPRKkoQMP4jB48yNpdFKPa7R0SucGY4ZiWlTfrpuBnQlaC+GoXx3NPUY9hjR+DqviV6rYAVY3WC6GTrtMsrYn5BYsNQb5uj4btfrMizrPlGLdvHIzSv7mEYkk1AFpPPhmvzQTwAnRKQ5s8Rwxtkv12LBXTXBGkC5yOfhxCtMT2CqktT5VSDsGa5d7gf5R9rnjmft0ID4YnHBt4aACYFx4zYOrzgyg8dcucNlp5VPTiJaoTNjK/G4iIdcFfCYcRLPeeDIWN85oQClBjgfWxMT96UfgdXSPNn6jNln7jZHshkey64xkd2Ak+h0Ci8oXa5NAztnpBFRAPjQ6RwT4ZOiTw6mYIQYpbx9/6E34izzMa8Qy2pKJ/oiINnsjXxMHKvnu1/EvVqe52w2t5ACFAPzLLZge3TMG2u47dhlWAeu27yRf9tPm6rZvVEHNHv+RCQYo0Z5ndRKU+d0Dt9guQMu2u34AVQo3XWRjzgcwIg461KD88kYkoYoftMKqOkBQBQJwYTQbIEppVtDai63kY6WzKwlphlmZ11e3aKKYm2BWnW/x4JF9irgrwm5nkODnKxVLevmK4j4+fQyBvhi6Rr0K2LtMjOOh8WiWBC3Zlq7yDVi0WIU+xvXP568Jr4MKrw1MrkZ7t4X671qswQVjmI2Zn7QtZP7II8GBJRvKAC0yaxn36sn/HIu3787HIgvlBzyzlD88sRfnr07rNbv5/xFdS1LQOFbfnPyCvyC1ki5/be3QsWdU9oe5sP6V/CtBX9Pqplvrs1KFNJRphC17ufQWUVdJXpiKVu3DN/Qh07Gv9aFFNDyrKt3FPhuLno8mAQCvLefzN2Frp0tzS1bY0WvYLwGLjgmTafuDT10mGraDAaqT1pefoJjRSnZzuFuPVuAund+0e7/p6/umZX/Q1Mwv4WgtkT+po1OPwhv+Ek7zrdONz3ss+aI08NhrgacBEVla/YF4rhHeDBgcMLyLKw/xKLzBi1sGT18shD4vLh2ATImbGTvt+h4N1Vcdq4kZNXFY8FZpadRJp4wzcHoKQriBZRqCBA3y7szx9IOosB1utfL6oCZBmcck8+fKNa3D374X+nM2P8bV5wM8Ml/mdGyXXWon6HTA/VvVUi1W9ahNqFWXULRalVhEd2kWD1nQVZrcYcLtd8PwvS2qjzPPOcuNv789f31h8RUd55dt8QHlrNfwrjkGAlUxjTBjRoGcHANweKwgpzV5frEMQ505pZBojruSjWQM3FzaJU8FH2SW2DK2717nIawz8mKnL2nnp0cYgIkzH9QOdsiA5DZo+gMxoG1ZfQe0ZYPg7lDq/EPW0z4dAWSd4/FSja+8VKkkQZjPuXqYzT9kY0LOOcPpSu7nV1qG8Uet56udfkAa+pwV9ZYZAOZ4WGw81mpsqOncYQ/STkm2bdkVUjVf3N3CF6nPNl80CO8WxnuIk+6cAWa062t05zhDrOsujHV3O2MdbPEu3K6f6+3uwPVuY3rGsC6sigkwLpDk173odwPRLZN57rpFGkT9NWjaII4L4r482GTV/RxE9YGrO7AA4N99Z4JV0nGjGESlpO0anOIOxP5F2994PXpCHoOuNG2Sk9Pisn1qcss4VF9GZjuz7+PIps6yOpDHstrRRmfsFDSHKMIkRLvBXwxNr+PwvetoGvDRqU50PlQxdIZzK9yPJenZ/0AwhctYlps0qPeDmitnnLoUI3cHFo/MHW6sZpAdKtEZdP8MMzp4DkoQgo/rN89fe6dnT88fvzs75W9dItuP8Mu/8ykh3odutiS5IdR3uwGD8fBBTCAja7a77tgoDtLo8e27V2/OTmnpXPx2Dw7Wti03zu6jYqfOxbO0g4ONJvza97/6j8BHT1zfjDqlPyhi6dLdCg8+riHAG3/wrpGAlUswnZcKkEhDatfBb2FJh2NHeDyWaU4e/aI+OI2uMBcgigdoY5KTMj2p/ovn52cAkpdnb9F2qHfvKLfDTFzU4SVKJNB21Ewd56fjMFyoX/tpeiZcPJO6boY0I2qFHF+63FBLMPibffUpRmrukdkcENpyG1F7XGKwIbKoc32sDLZ00pwfFeCT0gdbif3ML/iQYTKBsRk8HLtuRr8YBA276wKdreL5eGZ6QJB6gAM64wI4o2fnT4R+fef2ruVCYbOL52lb2Aies56n/IEteH/3xkBZcVeLG2qsNbinT/5xh3aU5QT1f0Q0eE23dKbPZgsAv0sLeuF/xIV/gaGfCh3uPA+w5rCJRw+hiT94Y5MsGCwYDD9/dercuTHFI9ShldZj1nsMUrQcI8SpHEQ9lPUHvmjQlSyWMhmgql4MPjbaqWiJcXioFRWLILRCnzERAs7jjQ6l4Eho0HR+cVXucHvrEPDr5OGEFqrTXPV2oBEesYboH+xi1KSogPmp3vCjFBd9btyXurcMl1aPaqV3uTwcszd0jtuMLfax6p0u1eYW2jPS3hO5jPGo/QuVUPzk/Ozl6Rl+FviIRM3R0cpflmm+P6J8vW1GDykCNKHL193odp8rXzsDkVSzPSlyR0fcQYirS+CBG4AQ+4k7rcJLqGwp+MEdg9Ci861svuFmGabqBd84fc1VTlGGvX5CO0u4cnXPDet9JPxS3fU2Xa0ZTnGqE8f4kUoeY/Ad1SyaRjuUXQ8lVTSsbrG977MuQEpY3U9PY4QslkIaGkj9lWrVPGdt/9u8krvV9tQqmKd7qN1FdNgOFaqFOFknqqPWqzpcWKsVdKB284vDWc4fvFFnw7ivyVgAzMUJBIAtc/Pl89dn9Bw0UvP523enr35/d1veBPps1DauxXZF5+lMja9ZY3LkvHV4GQ4YNcAsd3k0nbxy6zEd+CVWuR9LCl1QHVWXU2FZ6bOeoh+4R2uMSXsGuxQtWDtvtCjsfwU/OLBy2Fqv7R1XkRbFFPijWZyiP6/Yf7/dWNeyMc2ZspzdE1CxaVOqi6fZZ/bUwTzn+/TljR4u3WOT0tm8Scvk4dPU1floBJxcHQDWja0phsYfcBtWMUxwD+sOulQP6HVHdjOPqY6I7v1bgqLtlgY+ktRBhgvkeE577bojVEfo2VdyP9Zp483j9OJiyNkCxov4Zc6YjR1BGxoWw9/DUnRAR7oNHxlgIE47lx1ohTLYDbJP1+UN7mUAw91i2UHI0PDGZ/gNcOSkuAEXedNkILLCrV0cnUwmk1l3mzS/Ho1G3tnLZ89fnukFGY1QcgO/8mSyxs9JQMF5lgOiK/Cto3ThR0JVG6lh6VY6m9Pq5s/q9ip/hBqMbuz/AjKvigo=", "blender_rig.py": "...", "app.py": "eNrVfV1vG8mW2Lt+RaWNu2xqyBZJSbZEm15obI/tYDyj2J57g3CE3mJ3kWyr2d3TH5RkQkBedoEFdnGBvCywSJC9QJBgsRhgg31IkH0b/5P5A8lPyDmnqvqLTYryzN0gc3Hl/qg+VXXqfNepQ8MwXgepiKf8I7sSE2a+jLnrhW3mCp+JJM3gZgg3QRqH9CwIUzEJw0sWMidchCziMy/ge1E28T2Hw7MA/5/GPBEBZ5zB1VIk8CnLApZmAYCA62d+mLlTn8fCMgxjz1tEYZyymR9O9PWHJAz0dZjoq5gHbrjI74S+SoQTizRvlmSTKA4dkeRP0nksYGLBLH/gLcRe3jFNmvEErvb2pnG4YBZT72A2U2+mHooAZiv0q5lIbflkb+/sm9dv3rERGxvP+AIwEhsdZjwL41jQ1TvuZy7Xl6m8+pJ7vrx6K6IwCfHqDQ/CBC9e8Qm+vNg7f/vi3Yv3CHu1x+C/B7AGi0kshsyMeBImHRY6aSxEhzkc0N5hqbhOs5i3LdbvMWoCi7UQABdgLEJYxZC9z+JJSOAMujQA2nGHHZ4cddjDHvzXYYPe0Um7I9s8Fyn3fe5Su5MOO+4POqx/vN7wLY882WoA4AbHDzvsaL3VGWAA2xz1VJcn623e8GtvQa2Oe7UOj3qnD6HZ7d6Lf3v25vzrF+/s56/fwtyMAyQ9oNWDV1lwk/Hg8Hl3cMCTBEjjQFzzReQLG4DORGLs7e25YsrsqeeLxHTbQ+o1jF0R0ypKPNk8gPZuaM38CS6KegqcUnsSezNrOrkunlgTXwRuqYVcFe9jDo06zP/TzYDneOo5oRVOPpS+zh8vUr/0GAgxFokXBoeuleCbGtDIEx95oscaCaAFfcNjx4Ml1bcwAdsLpqGFjAdUh59PkWDG8EnE07n1IfQC0+2waZtNw5hNmRcofHlTphuJay9JE3P9m3YO8guAOS1AJMBJwjWR9y38s/atsW+02+3yxKC/KQoi/BwAgkyguymwp5tceencNA3LD2dGR3cLwiGLsbFe9iRbLHh8Y+KU1dpzmCxhANjaNPiHLEkFcuLqVnbuA58TQvKhmFOgUsAhSjT4+/psyPb3Vwhj3CJetJHVWxfDzq3kzf199vNf/AcYjZs5novi0cDJFL0WXxltJvxEMAMuvsh7nBqqA2ouO2l1WOtPW+1yJz/9Dy0E2IrLpnDfat+y6BpfGiWAIvkhE75IQ1YCDNSArc3So3mG1AOd9eCFvOkwoigGwrACsvRVpD5qdQFeG/suvUzELAN5nthpCOJFzeOWJSUynhpfhfGCD9mqtcj81FsCefFWBWnlFwpprT7QVxq2bivDMpu6niJ46pVG916hDdeyMqVW4tW6VRi2vaLbALts7EY1LjoqA38LdCviYWUB6JFGXAUhP//+v7P3ngAVlAwZDZ9tnliB08pkmElSv12fEj7MZwNzKRPSllnlvdRn9qVP82j8eOLnc2z+eGW2lp4rgMBa7AuWpLHZAEW2aLWBS6BZwkwBIkW+ZJEPypSwDXOqTHTt83zKRoXXiDB3Idn9fbVGUt5UmJp0iANCGsSJkjW5PLF4FAEaTOOsaASUB5NBoSxlIAmUCpQLJQ8r3fiha5MA2NzJ198+hyczEYdrnVQUB+D+Egl5mQuVtkES+7LDlih15ZCKHi8sLxWLxGw3DWzBwQixY+F7Yik2D+6tbDBk2B5FaoAUDNqZOWi7ofmZSEGryM5o6GspYlCTAknT5xv7Aqny6ccE7VUAiC0TxXzjVgWAFN3LT38vn1HrT/8rgA8PHBjkwYIHMzAajarWzbWv700EWJZhIhUECEpPxAH8e8MmMf+IkhOwx0D+wlyBasG8TWaZCJomJiXuXdN6Ra30rPJJlT5uXcDCIg5L88CuJxyYxRHSggegYB3wdtNASKLjagL74ty2jOY81w4O9+FvhDYH9/Nh1UFJdEdZkOIcvOmnHx20zjbi10w9MnhnPEYSZWkIOh2GDz4GgkY0JFnClsLx0KjeOBmPjEY0v/tt9pT1d54Q2BhyfQV1quBUp6ceEt7BKbm+YYtPfwCaFjF+WkJxQbuJF9j4MaAGCQSRrJlu+9CGYPH32G+UPbIg2yTxgKQYR29OK/oN+NQassQE6yPBeejXLH+qp4Ey4kqPVUk/0Mzk0YwvNlPKz3/7d//7f/6era5uFSBlsBnfB98HSkLRJ20w4N6+fknekHGWpehqGDyjxQPqB4akJ3O6AtlOXljscZIX8MLJuBtnkXDpg3cw0hwt+DoADzEDELfKTlx64krE9jxd+KY7+gaEb4fNhTebp6OH6Inge/X4yovF6CsOaqTDwFuKuHq+SGZ0pSYP7u4L6Vazw+fMRL9UgM3dRhICD9mbxnwBZCti0EwlkQdiNgKo5IKDxQ8WD/oR6Dsr6kED2AV7nC7usMYN5dhbOC8wrotlUWiHFXniekvQuTe+GLXkjIcr+e9tdP3Y9RJQrjfDqS+uH3NQKUGXFMDQERhNeLxOYQba0970pqtcNN0SIXRdwJ2Tgm4bOqGfLYLHMx4N+z3oaMKdy1kcgs4dPugP+of9kybYE/JDuujFZ8mwfwQfAqAwHj445acu54+n0Gl3Co65fzNMbsCwX3Qzr/W0AVR53vRV4n0Uw6OH0XXr6f/5z//tr58cQIOn2OopKlATlhexbrzLJhji4GR4goCPMj/hrKmDiQavhihOxKkQracvBUhdHiNZoAx+fQbgX2jifHIweYqeAHwvB0B/NddhbCKLfdA3sNBxkgcofsjCVFCTH4BhpkY6AgZPTYx9WPjHbLN9FBlg0ueEhCRdEMQP6K5NjT9Zjlb4omiG5F5rZvwJ8UA/b0NssNZIMgcij4Zn0r2cSBI7NM4DGZCxeeQdoIM+WtVJuFUmYbDA/nT1gxpbiYQ1M8XOqLWCv7ctjforz03nQGG93zxeJ25JTcNejdKaqGydPFsgav3wCogn8/0ELBcRPGbXcTeB4YPu66YxfAJSplUDLj8rPoIVlsOvEWmVM9ES6hL/DWMc/mNQgDMv6KZhNDxCkn3CGUiYaT7/FN6LdNQC65sHl60agRpVyjyZnEym08cFG/QPEebZJPZilFcRDygshJHAyEfx9PNf/M2TA15Q597eA9b9Bf+xmWIK1LlEKAlAfOMli1C6PsBraD7BGECXsHDqoYGB+i+PAIGPlAXSbgZdiONEde1wF20OAmdKRdl9mvs6cJmrhbbFnkFD4m1XLOHhEs2KQgoDs8PXKg76AGwCtgBzC9YZlS6IOIoGAjwuJ2Ax1DrwOkgWHoZuEJNLbxmywIPmyLgUJAL41t6//vZLUnVaH9lX3EvNJO2wAEgD/O8YLBCRJGXlEgnsmtACygI6mvAPIQMbi+wAHCuNg/0ZgvgztghxrOguQWu+RCMw1ytXc2A+6kqHW5J0bOA9mPx53AWfuYBf46LgdT0wE2Zv9qzT0w41m4Lr0kGt5ozAx4AnrQUaFBgWKAulLjZupT0wCK3eFL3XIv5D7RJfiAjgHufWE0IHk6o8hph7IApnsfUijsMYjX7kMbA+ETi0hZ6JRGVAKApNJOMCk8k8vLL1POwQI1WwtomHYXGONhAtHpCaDJnjwnN3AWqwUMtSCnsBrImTygDjJcYp66CVeUfNLDDYArRXhAlD/zJL0zCwHN9zLtso3AH3QFNJWdSt1uEN2RgnA6jF2Nyl9G4LKsLVkkaNmu4D9p2aH1prpIOIJIlcluLjEInDEX7m81j5KkBfLlqXAVuIBdAsLw+pbDehAWS8IJJT3OACE7wPkeKS9NMfkP5/+odDJAJtGamo60//PASsJmwCKjXY7Aco2XUmA5rAjueh/+nHWYgBb8WKQW4PFyuujCOKTkosRLA265HL8ryihlhn1CZO0E9BvKKsxMdP2TGY44R7tABVx2B9RDaNxQTFHQAvoxbpMF9M4ZrEODDLXAj9j016B14BnpfCnsw6BKMI/NfRMuVgnNM+ALQUAmaBDG7Ly1nmucjk1JW2ZuEmTHkq0KJ1RbgGkZ6CEwBQEwqjI3hASMonvtA7DghS+vBAgvk1WMD8UqhAyxpglzYWxOh9nGFLWOh4ZOSym5nxpz/ghkLbIFm3iNKRYVB/dow7DVxZ3HWwmhdGwELnWhS1S1KSyLEP5MWDj7wsJ28KIZ+7UECg4DegZ0dLLsp2d47ZQu7gHVCS3Kuy8B80unB7Y3//sA/CrS9pSgQzaFZsIpm5MCO6YF5CZEPURYSgnxRdKcPflLghQ9RoW0DxXmSWzPoGaVgzWJEXndhTj1y6iRzv0z8FVkn25lK9Z/X6So6DeyVIfQCiaG8lUF4LIhXoIvz53//XsviOb6rjknMdITYsbG+nodyjUZOSJFwAENeOgKm+oH9QAnPQ51unOjW+IYkVZTBKjCXFG8cKykHcGs3rUPRRHjL4QV5q0/KY60zblj4wfaDAfvCQONSGpZWGlyKw5+LafKhM4JRc2ymIcEAy7vSgX3ouPe1Auq2wBPBQupek9PIbUsNDtsYUSmeDWrjAVl7pk7QH1yXle0ufoeExhqFe4FjT3LcPfRdVlQ9cYGKT9njYHfRKOhcfWlEYmdCyQ12AvJWMPiXyAZkHfF6iTWUW4NYkvzbVHchdP+QpNC7t/OC7hWxZwAwDG9QxUIs0iNwaZJwqfuFWnioLRscf8LY0zKswvixzzxrJIoiYTCBJA3EWfLYo36bSVGdlUa9lO7lx4rqtBXwu1e+Eh9ETokoS//CnvUkJlKT+3aPcohXu/FhpipFcc3VHEftCn0g1iixxJzjaxRghhgh3bfZkBE5uYSyP8OJuMFJjjSZhCDZMocIAXZrmRvribmBazxE0ddO++zOlEFGeh8nMkIZubNJjlPM8TuWGp/Eem+SbhnOpQY0dkFWoUhpbcV/dc61wU84A6tF9hLPOvwA/WCC3rHVCVjyFAVbpTSRM0bZsG1nUtm+lfF77hIDRRzlYS7qHYOU5ZnUiUSxJI/+seD0F49T3G7hdujcAH+0UKSnyjBLrPV2ZyrVG6QFyiAOHB2TWqHVSoyg8OEPaoyUnTkoghd8CsXlEQ4oYkMqdamBSxSGNuUcUUrJsZR+0oUwb/Zo+HJ/fGCpmSabXdjpRIRtlNR2Qrf5Gm0YIHL3h0M/IYDBqdA16OItckCtmAg61kwp3pKZea4gbbg1doLLh+/vM3O4igp9+DgurTRHlyqPxUe2FFh8tinbZHlfNTULubtbjYFiYimA/qMQaDN0pYLmZSDqdNCPKXeii5LWuWxd1Q+29skzB7RKBzK1yw8dM9cx1iETbaSUC0/tku5NYI1kBGKNTpYGBXKD3eiPu7hVPm5sWa14FeY9Vr4WwzsPAExTOyPc7blhp81Sao6WlJ6a/z8IfDgvYnTJo7Ch25uA4J//P197Gedn22trHZfEtqaFDuzXosSiS6OAVPlMtlmSyBrAyprktG0il8kATXD0AZbo1JaLt6orqkmaGId1oY+bF4eHDnpEn+NDjirfe1pYlgvuIXruLkXHroxdJpFNSFzw2xx+jiwaX/WOkBOH4AmPqeRLYZl6ALpuoWkWh61RdDrB0qnd7NROjUxHXEhUKBdYiOgLIcjZ1tvn5P/45299/I4UOohsp5I4o2v4+bafhjCv5T1XYoLc8MC/UgDWrTHjqzE05FBk5UEbiVp5R/il9tpnKyRVd8tgDawG9UR3u2+AgwzLDGPwQX6I/M77I/ROvIylRBNkCGAPWirpuNwQmPXYAdnqgGuRBya9C2sAFHPVvD1ZFg9uyGwykhAYK7SUnXgDEDKa3OcW4TKwoa2qhybLZh4hLvgPCI6SOmoxzwvOI/laZCfGgXRhgWKT+i2oLwFFpjxUoJt9HmVB+LpAJ3rVvKS1t/WU8bgHU1kVl9jvYeNV+/9Nfbey37G8rvpOrW9n4DWeaChc3NunZxCyE8kvuCyAdjMRKHZyAN8RjlzbmTcoLg1UXS2gSymAHtJzGuP3YLiI5uHtJ9JTTknt3dqLMDba+/e49ZqCqVEX07jA6LC2r9lqwRkPwEikhq6hDIe8FWUE53mK2k/BFOQpTteVUrSiYGSRQ8BlZjvSo3Wjo7ZTAWZG6OpNiMatZ94hHvfYmvO6wtaV32+Nh//jCikXkgwFuGqilDJZneSpCIFDj4cPehVr8EIDK5TfFMh2iAHlHkvg5T7lCI6CUltH9NZdwbwOu9AqShuLBjXk3EvP1utMxy5NMd1tXleiKihpxMAYMWV7giuuLvQbau3digE6sn2ep5xexFLq1nDC6qcLSNzAUWnKbVKxtt9f6wJCNHKG0QVYy/lSRl+oVZiZbfgiuFlLC+uCrKcyamuriqqQKeJL8/2nf/HFNlN3Mkgq0+xpanboFssHqmKNhA40UJX5E3r4vPxPamnlaqxwQM8e5mEFXNosdkeuYKi1KRoiSCiPooEYI5n/+cLmA0cqG1tKL04z7Nu2V3ZQiEijc6TPLyVxuwXsbDR7EikkhsNJLL7H5Es+MTHy07wixuK/QKynnJoOpmhaxUihyVcTfFFbUvkV9+/bsDSjk5cLKEuGCeYT7Fb2h1Qfb8aBh8w1AQVuZIps3RkPz5ZeYO7qwIhFjls/tbygH9+X5dyxLuMuHzbDM2bh/AUbrbNwDYqn1vqKX9V4qCcMFqHxrnhLzHO7MBcwLbTlh0V1uzzTaMtdr+VAYcxPXSJT2q9fv3lOEPtXR9FmU6ctlzBf6unTpFC1wg5+u871YGTnVtLZOXkggSyQSlAPRFfzpWb11ukQBWRx7IrPSCIBvPd5NFh7rdsFhjW+6MNgRgvY+cpyvBfcdSZO06PqaFrWzfb8Vp4I2dhYLAhOFIIssN+ZX0JuMvo2cZNkJwrngGPUOwgx4PNkelEzmwvfVdqDDI4Ruh1kaZal6iIEEfQl+DrwbHWKIzYWrHDCe5RjLcPK1FODXdFxFb43JbRsTrEyjDeSmbztGu4hloosDfUQdiXUO7TqUX7kP10SMuGHBxwN8DOKFjw8v8O9RAQLpwU6vUxnNVF/1e4MjIuuDFX6cP5E0bdyhtEogDdx8B66Sn9wpbZbcT8g6cFITyQDnB8BGS9wehn9yfgXEw2sFCi5t9RxVAy76KI2KbM9LuyNxi+B1FngxXOKX8aWd77VcFyIJcxnz9+Nh97RXCUYhwE4+X7ha4EpoR6CiLgpfIOZTTOvWKTXmaY9iRG/DDyHt4boh4/4Mjy0mMpnS9zBlA9hnzm/A+PaLqF2FvZoHA+jM+XdvB99IH0Oq5HBVMhVbT9/ImWHIB7Ms4Q6kPflJKqtqm2v8IZwk8jlPUWYFsMgGPsRzTAZ3UoxN4dYi2mGBjfPFO3XC6evXb8oEcvpIUcjpQJLIaU+SxumxIoSTQ7VpCQ6YPBtlGkiQSirCPw+OJ8eTaQ8vf4PGnmn8FrSMoWUltjidPJpMp7rFXt36Vc116/701H04KcF7JvtzVH+ix3uHvY3Q3qP81XIY2x9NJ4fuKV7+9I/PNPv/rsNewWwoxfdUClu0MIACnDm4p9W4A7qfadV1lHuRyBqA4Q7LyHgkHBW0gBGEnPoLMb6k84dzMgrn4y5IjKcjXJhKK8yShKkeH50cuRTyp8/IHoAOyw4avqh6aDTc6s5nqT0qyXktCZ6mn+KsDX1aBKTZ70AOYgzl5JSEWWf1CnT3K3iIGV/XJCnbJNukDt+qS1To5roauoHpn/aGFzUTmxaiiC48iUL/BlPDWQQDS5NRawVjvW1hzMkftQKwazHrMw4vgdVWmAK5ZSRTQ7bsUn4otB9Yx1X09q2Ht62DpyU/miiiNJ4k4oFm7Yi7uEM0PIyu2WmeWZpnkA6qGaQb0/JXLbXWh4et6nBaDwZ84B4+bN2qhGeYoc5f7UOnSeiDb7MZsLMJXOvpCmnjlj2ZPF0RHZKCWmW3mIn85ABnqbGAUgTdMxAzMt1BS5bcMoVOqNFT1ivIqkKIZJb6/JYMttDXZ3v+qMhdRywNcgNiddO1hip/+7YpnRz6UCH8hDCZ46ilZLE8T4k4RWOWwGI7wsU6pkFZ2UTraE88oavr/qjVa7HrAVDr74C6b+B+hWzYs/p4O6jcakZQczk+1o+6Lk/meIzmZtQ6YkdI4tQjmm4YSG9IW5fYePrz3/4de0fnK9ywfqiMllhlrVP+OqBOisvvC4+FNiQrK6t6kKIeevib38t0QXkgTKNkb/OBhXJa9qA3GAwm60xRXVA5qvrS11OvG0hMU+EJAKYHmC1NidIsP2jwuJbpvy2ju3Kyon5mAibuiO5EpFeYVk5HJ65iHg3xD52ZoGMKK1y027VOGjCsbQ7UyeD4KeuG8nHlk7orCCLgoA54sxOIab3Rlbz9nVq2shFTHtxyRruNX4bXQM+sx4CY2eoVkCwecBfxUpxRduxbdFsKsb4hmf8RHh2p9yFT4xFFrAdI0rx0u2q1JLWSZkFmhMHsvkYK7bXVgA40WJJh7dvauY29vWfvMLUbrc1J6N50mCUPPdBiczyBx1aV4wWP+id9l/0r6SFycKv3Gr4Al6Sr8HHcAyyU2j9mFTK83Xswd2M277NVkeA/QApmClM91SRiq+ppGt1ggKhk2CzxXFEb7qTvDo4eszW+U9yl3+TMhDhkmpvw7A+CxeSaPwZcHK418UNM1lK9yNtqZ1J2VHCooCuEyD4ri/IAswrLUJDIeNyltUIv6rTnillHGcadB4+cI3c6bTct1HQ6LT/e08ZSfhzjUW2B1+VT7fV1F6S8G14NkcUGR/BejeP4GEYee7MEXCILCZitFGHfMv3c5xP4u6p2MqgS2e3eNAxTpETNJMiqlQZI8WpPMfN818487cBtcGrO1VZZpPJuxIKiXLFFl4n1JU8wWdTDcKI9z8TI8ALXm2GqZCCyNAZHmJ4mPliV26MPiNvRuAD9MgxnvvgKnpqyTg76HEYu2FHTJTxIumjc49kGKwHdnneAfG0XlGCTSWooToZvm97bLo8vS40KYEieDdAkjSK0pgYanG5VB0draRO9YSsi5wJW6W0OSDWpTpISdzUUKV/0/IpXGoR+XxsLkVcVkpQ3+XhSL/VFc1+lvYIoS7chvbHBZqzT+Qs7ESBjXaSvhgVwBtNDGmS97faR3gU6x/gO8Dei1pkL5xL4XmG3YfR6ybe33Lj8tc+aCWFjo9q4Jb9PBCXNlIoOoYV2PqaQgT2bGOQTH54o01HWL5Lnlbx0joLhSySXxCR6GRln54zyJPBUC27v4wECZO8R/YXRJckIlHGjYPiQjAyzPXq6ckMH/NIgtZCoLcfnSfK1l4D/4rpmC2eBZTraGO9xxaJ0TFsP6W14ZQpfLGzPHRmgVY3avi80eQNQQDajd/2A1Qf9fbAWtz+vbMtXOvoBo3/qoDIlg9d6u/+pPDzsh0fcP/6QeSJ2wwo43fczOr1rJg6Y06PDDoYCbOlKHw56HZbPH/VvHQFlOO/5JDEb3tfamIY6M2tsaKsSPvBoBCkMTC4DMjSJDEfGt/Kgw6cfJ+XMvZCZVNmMMmrbO6TQVnrDHe156IPcHBkvPlhDxtlCgOJfgn18GeCCyBkkno9uN48XYXzPPnBvaGS88+ggVgbcQlkNmAlOpxvoBEf9dINxrx50gP+akuf8TMzodODhc4t9JQ9T4qw+hHQKFOwq/9PfJ9Y9Z0HH+UcDPB50bcubo/ZGCPrIBSziazocopbwq0o+B3L2TQQsH3n+fYZD2KLUgpHx9uWXZ0Z+yn9w2Ns8KE2KZ44DmhKwYhpvsLYDbc5jKRrweHyXY5mkWGVOzcMPHA8z4Q52I2du6gPZ+o6mJDtBbDeg6YUcSRU/W6bdH2yZdr6CYtq0JOc0WZQU1q/bH52n2NwhsNyv2KE8cLXe2StYQCQ4uca1/u4F3NbF5qCL53EYkei/EwljeZioQ0dM6LQJoQUtUflCnjqhN9hmFzZQX+I3BZy8A3xEN3X4xsUOhyi4j6b3pjFrQQyYCFTVEuQQ4148lxdJYOZbb5ZzVy37agNFSfy/RT/apGNNmIKHORBy4GeytAgdsJWDbT5wuDV7Xqs96UcZ22lDHQ6R43qmDCdTbnjm3Kwn7OYmUuf+Al4pkudCVSjCkimYwxx5KKnCD3QDD11QkJnPP0eH6EI9Dx+qknG0Myc+YneDw/b91vnMp8xwUHH+PdZY8qRE5zsfDI/Y7Fmgdo47rG89kkdpR/I8o0LvQqRxmNxvbG+wNAHOSpe5uoeILx1y2rLoqjwWTZ9yGjeUw7qjlxoiYNYDC1DRs05yVBSYQJc3AGONu/fDxjMYIn50DyTQkaYyM443HAOWJ53evWQm1VHSZ6l2EUZNkqm5EzX/NzQoNC8w8nxfHpPspYeLZSfyUyJ8Aka0tFzoPTt7fcfyURg0XZNWqjhsLrDQd8qHfw60QmUs7y0dtGRQ0mUo68WSiSyPRB/3ByAc+sc91uv1ZIk4QCB4puAjYebt2rmMXUUFdUTlBhYAkk6igVx/WOrncYnmsZ6Jy1Vx28/rkOZE38tCdxaTlWeH7FhVz+3o0fhUJWz7OsHAaip9bGB+B1IulrfFf7F+LZ6ykEuGyUVqGce4fBfjw4v2fRdMrbg+PWNG1+27B6oO+tWkjqphVYMomYN/lp7JCepVGAcCi5MgRsqqABAzVGv52VqGiugKqcIwCZwCDFj0BZzRQGZ9yNfBpx9Tz90q4T/D8s6P9G4T4v8m82CWGGZ07yIktUaYALUNol4fWTLqfmL6t7K6phlG5NL5HVmajsj8Xk5K6XjsVmLSJa5U0zfnR59LT4qgvsG8HQfkMx5CHJYELFbdi+UeK7uhKuSCpaGLTMwTVqnIuXfnIeAaQ1Op79xCLMp9q0mqSp6f/imgBCOCcBcv6kPT21aaVothEi7D/OHNECepMiRk8Zg8StJcWcwoxWVmIkBj1/sI8/JnW7ogIbc9Ebdcm5vScGWaens8PLnYCBeLhl1vpzU8NiRLeScmFjiheG4yGpOLcVHYyB8EtNnI5JUo0hsvkcG2ajXLtZ7tNcyeOZhn532k9Zd4SxZb8DbjvgTwEo2W+EZ7le9DB+M2oSwIyrHAlh+qqpI6gfsenCKr6FF0Rbmkxxh/oxJj8iCBuFKMvRHoHNMu5WC/8nx5kMp2wixIwULGisuwAgXR/7vX56VjN7sg/WyJpVdQDA73fgURTEf9K8btIeUgdVhVt2ISJ9m5/c9UsedUUN/8+S//S7+HB8KpjvId/C2LN1RG18ekuodHa+Pr6/EdDgr3mD7f0gcVhaiC15Xy5b/VPga6D6ynfx80qPE8I3MPdAXaSbs4SrusnyyaA42/yRYTnMHg8CjHwDux8PRheeFQ2bTRHTGWvDDPVo105gueYvWqbUyramtUEYw/RwBeJNZoUX7TcQ71K2hCHpAsoIXh2S3wZdWNCvRu/wTg0h8NH5MvtRr1UJonHlt4KC3SGLW2+dM/tn+dpcjLftQN2SMczyOUJP0eje/oqJebsbg/oz+8tzNWpa+3RTGBndSnKj5SweCR+vUJGJZ6bWiqL8hK+alMVlu/n/H0dQhrZpZPrd7LZJrY+pDyHQKWHmOQEXScQQFNo9ByX5VOy27uaecwVz2+dCfkX2Qe3Ql9Td2e0+kGrN8EyL/zc3lCuLyxdtcX99V425h6ffS/FTHLZFlqzK46YDJVegezIS5GtmEun7GhJxJHwMp4O2zlPSrv3WG2TJPSxl8oyZLaKDfUxd3fVzbp/r7FAC9YGz32FiKm2h07uoCyikhH7U+VggI3WHmTyjqUQy7ql1x2taNw71IZT48aVE11qxL3fcGAT3bYsuznlU1wg6KohbJZXCxsLNciEfvq/ZuvzbXqimc/ZJ9+ZByMR+EI0Aalaoc7WGKDcikPOaY8pLhtVHmoY8dBNRZJ2WmETcUh1Ujz442bR4oj2zDQhs6B09ZEs2L558AyPMaK/yZwv5wF5qV+kZf9YAkITqkKdqKyRhGzmdgafHddlJR8XQ76jf30D1U3+6d/3kkx7WggLOknkSRuaAzmulwvdk9PthhpSxvc2AZAL7WHux1MUxBeHZ3Bmqzn34GAlWdH7topWOjAO5FH9WRRSfAtwk0aC95scw31YaVmWf8hnCjDBa1A0zBUJTq5FzTFusb+jS3Dv/K8Rq08FUwQrIQOQwWh7XxsV/Wy9UlG3RqjjmmpK9mB5cx5MBNmudMO0/+Om2vRASdftDcA8vli4nIWDBmgZ6STdQqQpXJj1SpcqPHooA2ya4eEa0dJwo6UPR1Fix1JSZ2CcztKF9Gp4gINJqyRrKRrFjVYG/l0/EcqzFo4a79qddaq1f5rl2gt70qqEq26GGt5/S62YhKovFg+uZh6jXDiF3hSKnCyGCxx56aIRe01BjeKhnR6D10PrLomYrLx9/epmLMEXxPwVpI5eD7WLBf9UqMbK6oqj+4+42L36JlIXPVbkPsv6azoTUkjRellcdZRNT/hWekAeqmx5NYhM0t1VjrFIfy2BjCe4egpQlTifHhoyboDZlGqQ39SzLL4YFLqmuoLAeSJZmFyWzrax7igV/B5Rxr2GxamdN69BDs/0a8HQ/Z0+SDqIpSFJT4fAQQD3KZMmCC1Oehy+y5CxcQnUgg6XUTJaAS198tL51Np43jPfvfi7W9fvK2WkZ+CdLIxI9vEP8X52Wce/iwNljKn8vG4zewI3+UEDE2vlKqcTbKE4pVgzdM2pu9N4rWy50noXIqidOylLFLjBZjrhbtgV2JSttowTqPHShVkLxvqzbhYMg0LZ1TLpq0d182BAgWEiag5S5tPV6+VBknrozLSLAgEJiAUg8MqDc0DSy1V8jzVY0BURIiKmPQjYqrDCF9fsKNeu5Y4KnFoyX+km5HsMO3EmnhYf8foWfQ/VLgNpq4uat6EnG/fUXGuddiVAkUaAoxfkZbPs8CZ23KZaXqjRycP9cRooqM1+ss5CN4VhwIKvpJATToHFFMt0FFpauox9SXxKXtXVQLwXJvYlLaDwXARpLas5mlj3nD+WXhlC8SBeqCTu/pNMVMKrQvXpjNEo3G9Ckl5M0QJDE1VY80SVKYYWX9dFBDPOvlv5rr5gYlrUfnZ01ITY0vZH/isXCa5WjUCRNRMUD5Z9wf6+y3uEF1vSXky5mkaJcODgxmQbDaxnHBRGkl5UAdgcAieiOQAj2Ik6QHGCFDwHmx2+Y3S911AVHbd5Qv34RFKUMorr9Vogrk6c5DdOMsO64WPjo8r0hUeV0kV5BD+kG4Cktml6vJZgsm7HM+Ha/rtV+vZ6ebVWkL4ZK20dxb7CZ8K87T9L8kAZZpnPEvnIxNn1cnH2ZBggM3sBZABVjZuSprfnVd+EXtIxUiesXSXBseLiVFnGtQeFY4BRMMt/c7CmlyUfFLloHJFQKdaPOWc6kuNiYAKeW90u0HYxd8akzWW5CPodhdff0pMAjzSHzyidesPV7hUt7uEymVxk1F5hK/PX6CNCu5HXH7+7v1zQGqpSEqp8D7WkCmqQxZ6hn7jpVo3MmVP2MNeraafPHeMPalqKxaSgV8tM0QGKJYSFWAKYlTEjHPxMObdj73uaffii+8tWJ5iNb5HkYGRVexC/oZBveTaYl0RyQVfWHhKJTIbIgRgk/CivPSmes3a5BvTby2gerZRPZfmebGlnHOVKBWtXCg87VISRFajNoofRK8V/kDhm1sZOOW1mrEKwneJ/hEfEeAhBCZ/kj0s/UINHWuIZVZ6aFnlX3YgKVO3lZoF0ppQIlkjBUGzqNksORpJ//66tCAHGhiNCCVvwxKtyQ2lFqC1lPzFwPf+Lyumie8=", "estudio.html": "eNq1PF1z20hy7/oVML27AGwQ/BJliRTos2Wt7Sv57LO8t6lSqewBMCRhARgcAFKkaFbdL8hLqpK8pe4hqcpDqlKVh3vLw/mf3C/IT0j3zAAYkJRWu5ts1S6BmZ6env7uHmiPH/jMy5cJ1aZ5FI72jvFHC0k8cRo0a4yOp5T4o+OI5kTzpiTNaO40Zvm4edgAYD4ck4g6jXlArxOW5g3NY3FOYwC7Dvx86vh0Hni0yV+sIA7ygITNzCMhdTqIIw/ykI5Os3zmB0zrvThuiZG94yxf4u8gZSxfNZvuZPCw0+30OofDZjMhMQ3h3e343X14D4OYDh52SdfvHcBrvsgHD+khPaIU3qIZvB2RI58QeCOeN3jYd/vuuA1v7AqwjI/8A3e992jlskUzC26CeDJwWerTtAkja+SJ5TJ/uYpIOgniQXs4pcFkmg867fa3Q5d4V5OUzWJ/MCepgZSaQ4+FLJXvQI05HANbBp39ZKFlyyynUXMWWE2SJCFtigHrnE4Y1X54bWUkzpoZTYPxkM1pOg7Z9WAa+D6N13sPc5asEpYBH1k8IG7GwllOhzAKxCSLYUjHuXhKJYXw6AdZEpLlYBzSxXBCksEhDJIwmMTNAHbOBh4IjKbDm2YQ+3Qx6A8RsnmdAij+Z71nZ3SyqqHZOjUXiTkUfBt08KAsDHxNzKKAislmSvxglgnaEuL7yO9eIkjrJguxnebO8pzFlu3m8UrZLU+BPQlJgeQal0HK5e7tjZ2eKBvBs9aBXYREgngKjAZMszQDVAkLOCeup8CYJmzj0UHMKh5IomxWI+lh1+uOe2RL6LU1A2AfcUPqrxigDfLlwO71i319OiazMMcFyQr5O+jAs+9Xok5pSPJgTvnwaJMp9xfBDhp9X7MjGs9KAccspsNtJRMa1RbK1tml+b9SBw7gOQpi4SwGnSOci8iiKa3toD2fVhZBZjnjxNssofHGCdyQeVcwi4MascSvkEMdZij3QkPO6SJvcrMYoBlt2zAH8KnHUsI5I/gkiT9EvUKK6wfEMyl84ktKLb1DBUviB1M8cu0IYmiHBhaLsoiE4WrLOoBvJB9wMYInoYvkFj1iV6WYVBw4jBQ3rwt5tAGNy4CkaIdPEhMDbmvcLfWRx2i9Y5ZGwo5Bp+nfGU2YMRXvU3M03BX5QUo9jh4ImkXxLu9VODbUGCHVo4P5NVBI4iDKVltOECVT93PDz7MsD8bLpoxhEnOBolCgX213iqsq1KS7YQc/6aI2iLJ30QWhbqcg+bgYeDgejwFVEI/ZDhGKaAJBi9v8AdK4rVSoERA0qZB0IcX9IZ67MN2O3YddspxMqEv8CV0hvj5KSwKgNSuSe9LmkgvphMb+6jZXxDeUavZkw1ns9zedhWqGMnPwvF/sq0pz3zh/zYeWzCjPYoN/0/xgXvO11Um3ZCzGaytqCrzDDmSSAsRo7XWBIFgVjq7i+Q5v1VNOEMRcgtKVPgTfsUsSQQw5IbiyGnU7qNptWtvqVLCst97zSDwn2aZP/01E/YBoRqUsB/twEHO1p/HkqOAMathh4XoOy4ToEPMLbSvBUE3vqC7VHl8hTASRCl0SWrd7AzWSyHMq+DoVPs1NNxRBK/yp3KDXFtA5Y2G2qru2Ur2bCxkNtbqjKg7V36Gr8lQbJqmerdNv1w5VV3RcLJW2Bl8/qGqR3QOwyOoE+/wERSp01F3vrfeOWzLpP868NEhyDUsTpxFEWFpEJGmMVvIlawxWjXyaUtoYNKZ5nmSDVsvzY/tz5tMwmKd2TPNWnEQtDvSbtt05aNvtljsLQl+M2RHzZyGFFQ1LoGoBv1ictX4eSrogEaTxWetzFrUa6zUcghMPp2iJ8glrB3gDI9YC32kAo6Hy0TQ+4IUky5wG6GODT2IhlfFpABBy1HySk+YcFtJFo1jA4sboA6jZLCXHLQE32oSHLBkYNXoX0BuSlVA7UYuQjsA0Y9mtGGHzZWP0LPWCMLx932kAMGFO1C2PW3DarVMDI3ZO+L7gRohUNcoN5CwYbGP0t3/+k3Y2u9H+9o//XRGioMBEaBcfQ6xsebXZKMrO3QdBwAXYbsBSgJRPmgF+z7yDl7AKfsCnehSWPSseb93CT0kE2aRHGqMX8Pj1z/h8KzR4ihgg3+GPZqRQW2ck1XKhBxVZgqd3ctYjUbSbsf/zL3//r9rJ1z9DHCE/m7ue02hb7cboe6zP6O5zAEznkEOdQoUV+uRWsCMOdUZ8pgU3f7TvYDsANxVoCGn2rVh7fatz2Bj1Wvu3grStwzbqeRq4t1M3huqwMTohqgX+NOMh7d7N99MFOjYQ5508L3AIAdT2K34KP4MRm9M3ITHwBNwdDdnf/vRvm3BVDGho3P8CsfepBGv5Xr+WXsn+SpVjbue2O/Lfenga8m7MFMR5PWhrba2DKZfEfHhYOo7quDwabY9joN0eFUGr8sM4xoPnLR6pdNA8Em+oPo6nLK8552cQSpswSLZtn9tfDu7hBIwkVQC2kF5D3QPmzsKv/zFh8Q7HzOV3RUO0pT/OaEhzthtmCuSBaf/DX7QTktSiRuWci59a8BVREg4sAq/2SCOZ9uHV+9NTbZyySNN5NNSHxfzqbeoG+QmIMgVGrVWgIr56crJVA4U4q2B5efbh+zNQYMiCd6EI+VTWqsDqy1+8f3by9h7rFbg6gveMRafxPEhZDKaW70RCq/mstbFAYNuDo2a59o2TOSOfeTM+A3JKl+cgKy9nqZGZQwn1/euz03NnBd58cKELa/14TV17Erq6VQygjoJJ1AdFAAhuiolLi4d+QJPw2C8HZZDHYYzychRjOgwREdTl4DRAGggMfCQQycXouqD0d69PTp3VJo0D/eXZcw1CNk2ZBoOaARHNY9pj7bfvTl+au88gFskRzYBllvaS+Sy3tA88S/ucKSvB8dhjdwGrvn/+d9hq1mih9prxQwyppKX9EKeUhJb2JoDEjMHiPU0ut90QPAQsfo6/FIK5GDFv56QgLwuUjSpgyCMwdDObuZ8B8u3z3wJUzkM4U7b9CDqV0gw8aM+3szwE0PMPZ5oc/fpfsdZ7AfVMpJARMr/zsd+WJJy9fVGwtd/+tgbV/djpK1DRbFlAdvrfchoUDRjoIhXUQMW16Qx0AHBVqjDQ35CEQOzU+BjMqToBRIM317B3Q5CEum7gWhjQcECDY0Gc8PjRDORdSn14TfGA88Cn7CNkH0FMUjtK9mHpH3BM6ABfVIJNwJp6kGSrYPD+1//kEMD9jwnmQMRO4gkAvI4gkMV4AD6piUl9XdphysVOUyem18KH2T9S9+XZezlurEiMFxQByQZ5OqMWSoimcwqKfA3R6/lsPKYpn1qD2RbooJrM3wULGr7HvqDxhuRTrPMNcfdRzVhd0xyqi84h1hlBHNP0R3k/Ao+vuJWq6NksT2bgJyGYnmNL2hGkn79/+bwarBCDU6cgyQQIlpDPTk7Pvw/CKPA+VHMKJTzIwrBNY96idvCEw73SY2EBY8MqWHAyhQLKKJf64PRCikClG8ug5KUKh8/x3TCHfNyusgQFhJ/CaC/E/U6JKYlSGilg7968P33zErCkBD1nQQT2shAc3DFURA5fZaO7Fjvj+g3nbJgWVG77pi3MnRYbglqq20GASLDtOKcnJIJNDcgcK2m1FGkhuo7VbbdL2r085KhqIc6ADaydvANwyfwXUE2i5LgIcBhr8veQR+R019h5QkFeHbtbbDxJA5W1L+H1FQ0TUO5O29qHFHnRpb2OeOh297teIRkIagYuLo8wDhlTTeUNzaaGwh4oQuhLyiKap0uj24bjm5YidqlUWDeR0FBvPdYm70Wz1Mb8CFNLe+E0udm8e93qFnNQNwHrBR5x9IpQDlJSOqVRoBD6Cl6zZApMPkPhgGKN+T9w5B78s79v2QfqqXF5pb6zWEH1omg8k3ADVxcwzGLbI1leIxHGhD3ZEUnQwNHQgT37hxb+R90XYMtt0yC6e9vDQ0Jg246KABYBgj2ISpo3A1HNwtCKwN3IR8JRZM5qDUGeeVeqxeE7mAHk2Mroc7boweAr0KdDC9NKoTkCG+bRiIrMSSB0+xwOVh5AZFR8vErKjMI2fcwE+KSSccEsH0cOvaAeBLX0HSiBof/S7ksYuFmLo2xNwnzc0s2hIIvvoGzMYZB141nMmaTxToNxhW1ETZB85qxko2BwUUnduuha+1bv0qokAtpkdey2hfK/tIqegVzU9X2rZ8OyvgUKgMvcsX8Iy+y+ZR/hUrEMU/+ycSDWkvYBGBVAXDT3YWO++KD9pI17AkYwJP6vWF/2ERRie7DHRc/qW80Orh2P+/0nTyyUrQ1Ed/rWmIQZrOVdBWWdvW9dtGFZX6wSpgPrDvi+YtH64upyCERzG0D3jSx+RRfG2UX7UpgG9rBjKB2XztlF55IPFaUktwnbts8uupemHc3CPEjC5blHQpIaryBIBtEm1t6lGFax7nMScLSGuNl99MrCf5u9R4AMzbu2rH8pbUhJ4GH4yeVTEUIGqO/bIK8VFAd852BsPMB15mrHHo/tA+xplhrmgq0ZJrb47Vnig18Eh5cGix9ZCgEVhYj3DQvwGlf0NEryZWk77paF4t6IB4rHOQQpajBntAJimB1k6KXN4uX8CoOUL8bAzq54BhtLAgARAzZHkFzQ5xiSMcVBIocuDCdLQOFWwyZmAOFSUL0Pc1FFv7nmpSsFzYBtJjIo/ATuEuzemyB/ZjEw03DN9RpzKYjewFm2UPg8BkugBrlxwDZo6HS63KYF9yVHPQcxTWh+wrvzSlD7Ay/KgMWmlRVAPEvbBTJ85YhcjyyMzF5akAMI0QiXp04uLATI7BvzUcc+avGZnMSYENhjNn9UBD/Ibk2IgCUS4pCbchK7ZtShoTrAQcmGAXj24jGHgeTbIKaA91hmUPORb3n2spqVIzePS5hNeHEkSDrAOwE3hPTwAhF2jSlJHb8FiQ9/HePbo754ETr2LmWfRSgTMjVErlMqoGohWE5lYCIYz3Knbc2d9vAuNf/uuwfMnkGC/gI8p429EjAvkQQ5lXoN54+diU1y2N0FZcwqVnmgd/kwx2nexHoqf8VEqze4Y1mrhwpYyukeygKw3xg6v/fRTVskjx/enDmfvllxfvOk2MghKWVnDD+QOoet44lhrjV4+PrneDILWab99S/aN6v5DqD5139PIQDQ7NhNR9+sQOcA6HvIBnyjC9Nf/0nDweWuwZvaYPTpMVqzgZTavEzMnn4S+ypDa1FBZp8Gug7mT7Jl7GmlKOkCKrDMGJsrkIAUSuqQaxJAXklzb2qMrRWIZ8r8gf7q9NkLvTJnKHiu1h5BIBCoHORRZ73e29wom7JrAy9NlMjtkdjPHN5OucCpyyFq1DgIKc9khlDDGjLF1dhYgJugVzyzsacE6TZXHH48dFNKrtbS2eOYKSgq5IkJBsiTN05t2Td1dLwG1Sv1mMijy2wEf57hOQxECDbN1ZqCFHISmgpWrE9O5Ad1n7Y7uVpNdTrt9iPKcVO/VSBbf/tJ6ClsALZkrkRYgxKJzSkfWcNMlettACgTw42EcC2CEBgPXzG8OyCxrUyZ7crwGRRuM8i6ohPYAQobLvYhDy6ADoXpOI6OLSv9y5fidRroJo8UvNTYqFjOwc36kFdVlYjoSkMiduR63oEFzJtMY5plA7vft/CbxlC8tfuCc7WtZQ9NxxPde0fgCUiSV7mysaBs+mQNsUzsVGX2KJjhZtiKYu5loiC2l5zdpUtaNp0olnVTNQjuEwu6+gggrUqEYVUiDFHteAe85p10XSoP+EJs0HBwO6TxJAepSkzl8Z8VIG9wojxFYQa85e9c6O8pkITNpxPRB8InlkJNDA/nJJz5RD7l/OE5mCV/eENi3pd6BYUyH3jHMvrxg34pNqmRmLE0NwxiuaYzMvjGwrm/BTu38aNV8/HRkfnt0VFzY9atzUr6K5fhoctQdzJXkokXHl956XCu2F4YJM/4DEZLjqXgg+uUrRUP3EtOZR/A0EVPXscsSbV9gRkMBryLRw39I5xd41Ashn2gtDPgmOh6DAFq1sSptm5cmatx4ILyQiKXT4ungXpESOklVnE0Gb2x6cGxoXsFt3kODkJa7G3OQE1IixWynYYEo4sBeotPJZ0HEkad5HcsZxBeCh+lc4ah+dD8R0gCDATGbKAGjJcOJIgzAV6GY3FrtOXAN829cja8s64/1fnnKPpAxysyfQjHqzpDxwf7bbOGvCIDjVsHpde3KaiMbnP3p/qxW97MjzC+nxCfaNyRaY42i0UstnHmBQM9BlZoEb2BbWNLC4nGu7U46DOwHFsfcBXYPBJuIhrEEF/At9M0YfI2cqQ/5lBnnFjDHAi/IHJtqLJ59QxCPxO9S5nJDW8PkIJtgGL3tcizMISl/GMIrbAJSJTTUwJZgeuMXIWlOZtMQqEGlmvj7Swogj2Hk/G8oJ5j1k4hE5OMUtHzeUMSdLa/KOWMnGdpSpYAx3+NKjyYT6tnsKRB9SZda/TddxG3L8COtPAURIyYJh/ApF4MWPpD/XFRHE94cVykgKaMIjJjuoD6GhdfYiPKMC5iy7sEJ/HpmF8ZB8VVr3Jt+83KW+P1ZwDpY7wWt4GfTPszgzqBJ3my0YReXoQPha/oTuKCGVC2SM8SX6JtPCBF1iTSEMTw3Xfy4YHjEFM+Qwnh07ez3LB75pDY2H7PDZOPvo5x0OYbcSFxMshP6ZD6RdR9dUh1vY4Tb/ldYLVStUivw2Jedj/AoCcPK5hxnTkXl7doVV2VrqHKmGVTg5kgSXgpiL12Rte2+Oa9cHrX5q1dgAeFsn75srlBlbpyxdtqERj30eHBRfV8WXE0ckYR34U7BQfYIRP6dbldqR/Xm23lqlyz6jPPSRZ427nboXvojsdWuZ3IqpS/DRADStPZ8mmST39Mg5wORDFhllH5us4mkYpe26JT/5YnLV1ITdF3X6OVrcueK/d3ji4bhbpyZ7KtisWH+5taWIRwLAMo+EmWQNGckAmPvVUH1XGlCshkQY4zx98d5SDkiyBzN0kKNQtntNgVXjkmk9sxM/3NYCam0fWUG8H46RweEAivawydn1C3MEv5PyLHvJPXF+KLqcvbeI2ErEr5VVEj3Axl6/vs4/3EPvVbHJEhYWhMWX5nQhNgNlqQ5mGoxm+OoPb42d0sCZo725OevbBE43EBdcSrR3b7CTaEisR9s7eUA3huL+Hfm8cA3O+Lho7aHMo3ejyVGxBUXBALsmTlZDYkBQFkvxZwA4PV72aRixWnbOVZ9KfEcGemoMqCtwuUJAHVqJaEqcC3pHBFrJBZXCXIu4T+oP6+U/xqEKqDm2uxj5enG/sUGdiekvTW6RBeInYe3CMtHu6EUelCt65EPEmXTM3v2Hgred+x8RbM5sZ7ZRhRexHVlaNsb8v7Tl59VoBlzLJ5JPgAPlua4i4Y/PZd/AVjV72xU9ohwjzubHr8ynC6imwloomoFNkypjn8IpUHI9l4v2/jZvj/TXRBYWdYP4AMu0hzoTfTbcMp78fFg8GPZYEfKpwYubV6JjomjVPICZwdt+ygThjif3h/ZugBfirSws9GcIXPrmMsUxzdEx/H8Q9KYIKTZaCWb4c0/HjnhoqYtkI3SfjXAs7uLwTubonf96OQza9Oyn7LGWOQ41dm5+cOv/LFsPCChjkxZO0OpYZawvvbvvo27uPeewZvv/KNQO2wt8uvhUV301D7u3oaTD7yfvHnjFfc/Bd4WXZ2q4AA1Wl5ofzWRebYcxJCSSt60CT1psGcZV++rGTSttHB3fpUzfp1389Z6ldT6idS9U+iLk3YVXyOsIsqeZIruswM/GLO3AAX3AJtiIIM3AvEMSzZ4FdUbLLRLcxTwJZddVO0qgUuyZNfGh6rZovomCvR8dLOICk3gIZaa1y594gcNGP5PbJ+Py5sddp/hlmPVWvFl6ppgsgvxpePP2nH/O8foYgdF4mFDYdOwEJM/OsMPvkJfJPaFiPcj27QzslEeh9I5L+OVrWsHO/cX2D/PTcG8FPnFFX/HYEYnxlgzkRmYTjKeY1VyNz5PVq5oc9188sXPceLByRZ3n/cJsO5o0qGA4NiBLFvXDkjsfbqtrWwDW/By4As9xd5A2apHd3cmUiIekEmt/Kjrbwgn+YkQe7hTUheMNrl+lV9ob7VGaU5DGz0lkRrbl2amLgjwj6QiZlS+Qc6xy3+lznHLfE/P/hfWqCMWQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
if CARGAR_TEXTO_A_3D:
    try:
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
    except Exception as e:
        print("Texto a 3D no disponible:", e)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()